# Five-Fold Baseline Training: Preparation and Manual Execution

Phase 9 • Five from-scratch architectures × five frozen patient folds

This notebook prepares 25 intended training runs. Preparation executes bounded diagnostics only. **No full run starts automatically.** The first manual experiment is ResNet18, Fold 1; its results must be reviewed before the remaining runs are authorized.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Fair comparison and locked test protocol

ResNet18, ResNet50, DenseNet121, MobileNetV3-Small, and EfficientNet-B0 use the same seed, patient assignments, augmentation, training settings, validation checkpoint criterion, and epoch limit. All weights are initialized from scratch with `weights=None`. Architecture-specific tuning is prohibited. Valid strong or weak runs are preserved.

Only train and validation loaders exist in the training interface. Test CSV metadata is read solely for split-integrity checks. No test MRI, test predictions, or test metrics are accessed. MorphoNet-BT checkpoints and prior results remain frozen.

## Imports and reproducibility

Seed 42 controls Python, NumPy, PyTorch, CUDA, and loader generators. Epoch-dependent augmentation uses `set_epoch(epoch - 1)`. Workers are not persistent. Checkpoints restore the full RNG and loader-generator states at completed epoch boundaries; an interrupted partial epoch is restarted.

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path
import sys
import json
import pandas as pd
import torch
from IPython.display import display, Markdown

ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.models.baselines import BASELINES
from src.baselines.training import preflight, atomic_json, completion_summary, train_selected
from src.baselines.diagnostics import (
    adamw_memory_preflights, isolated_resume_smoke, safety_logic_checks, protected_hashes,
)
OUTPUT = ROOT / "results/baselines/training_setup"
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 10)
pd.set_option("display.precision", 4)
protected_before = protected_hashes(ROOT)

## Frozen data and Phase 8 factory verification

Before any image loading, verify Phase 8 factory/configuration hashes, frozen preprocessing source hashes, all 15 split hashes, class mapping, and PID separation. No split is regenerated. The baseline-specific data wrapper retains the approved internal use of training masks for augmentation safety, while exposing only MRI, labels, and identifiers. Validation preprocessing requests no mask or morphology targets.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
info = preflight(ROOT)
display(info["split_audit"])
print("Configuration fingerprint:", info["configuration_fingerprint"])
print("Five architectures and five patient-separated folds verified.")
display(pd.DataFrame(info["runtime"].items(), columns=["Component", "Value"]))

## Shared training configuration

The Phase 8 YAML is preserved byte-for-byte. Its disabled preparation flag is not changed; the explicit manual function argument is the only authorization to launch training. The scientific fingerprint excludes only these administrative confirmation fields and excludes architecture/fold identity. Architecture, fold, split hashes, source hashes, and runtime are additionally checked in each run identity.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
settings = {
    "Maximum epochs": 100, "Batch size": 32, "Optimizer": "AdamW",
    "Initial learning rate": 0.001, "Weight decay": 0.0001,
    "Scheduler": "CosineAnnealingLR", "T_max": 100, "eta_min": 0,
    "Loss": "CrossEntropyLoss", "Class weights": "Disabled",
    "Early stopping": "Validation macro-F1; max; patience 15; min_delta 0",
    "AMP": "Enabled on CUDA", "Gradient accumulation": "Disabled", "Gradient clipping": "Disabled",
}
display(pd.DataFrame(settings.items(), columns=["Setting", "Value"]))

## Training objects, loaders, and metrics

`training_objects` constructs AdamW, the cosine scheduler, CUDA GradScaler (initial scale 1024), and unweighted cross-entropy. The loss receives only three-class logits and integer class labels. No auxiliary objective is constructed.

`build_loaders` creates train/validation loaders only, with batch size 32, two workers, `persistent_workers=False`, and `drop_last=False`. Train images follow the frozen augmentation; validation is deterministic. Whole-epoch accuracy, macro precision, recall, F1, and weighted F1 are computed after collecting all predictions. The same zero-division convention as the main training experiment is retained: undefined class precision/recall/F1 contributes zero.

The epoch routine checks sample uniqueness and full split coverage. It rejects a loader with any other partition before iteration. Nonfinite forward loss/logits and missing or nonfinite FP32 gradients stop the run. With AMP enabled, GradScaler skips an overflowed optimizer update and reduces its scale automatically. Every skipped update is recorded; learning rate, clipping, batch size, and architecture settings stay unchanged.

## Checkpoint management and resume

Every run writes `best_macro_f1.pt` and `last_checkpoint.pt` in its own architecture/fold directory. Best selection uses validation macro-F1 only; ties do not replace the best checkpoint. The scheduler steps once after a completed epoch, and checkpoints store the next-epoch LR.

The last checkpoint stores model, AdamW, scheduler, GradScaler, early-stopping state, completed history, configuration, preprocessing, class mapping, RNG states, and loader generators. It also embeds the best snapshot to recover an interrupted secondary save. An already-correct best checkpoint remains byte-for-byte unchanged on resume. Diagnostic checkpoints cannot resume real experiments.

Completed compatible runs are skipped. Incomplete runs require `RESUME=True`; incompatible completed runs abort. `OVERWRITE=True` archives existing files rather than deleting them and cannot be combined with resume. Active lock files block concurrent writes. After a hard process termination, verify the prior process has stopped before removing its stale `.training.lock`.

## Real AdamW memory preflight

Each architecture uses a fresh disposable model and the same batch of 32 frozen training images. Two AMP forward/CE/backward/AdamW-step/zero-grad cycles capture initial state allocation and a subsequent step with optimizer states resident. All models are discarded afterward. No real experiment checkpoint is created.

Any CUDA OOM stops preparation before full training; batch size is never reduced silently. Unlike the preliminary Phase 8 check, these measurements include actual optimizer updates, moment tensors, and gradient scaling.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
memory = adamw_memory_preflights(info)
display(memory[["Model", "BatchSize", "Status", "OptimizerSteps", "PeakAllocatedMiB", "PeakReservedMiB"]])
assert len(memory) == 5 and memory.Status.eq("PASS").all()

## Isolated pipeline and resume validation

A disposable ResNet18 executes one real training batch and one validation batch through the actual epoch routines, using the configured two-worker loaders. A temporary checkpoint is restored into another disposable instance. All state components and RNG draws must match, and the next optimizer update must match uninterrupted continuation exactly.

The test also rejects incompatible architecture/fold/fingerprint identities, forbids diagnostic checkpoint reuse in real training, preserves an existing correct best checkpoint, and rejects a test loader before iteration. Partial-batch scores are not reported as experiment performance. Temporary files are removed.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
resume_checks = isolated_resume_smoke(info)
atomic_json(resume_checks, OUTPUT / "resume_smoke_validation.json")
display(pd.DataFrame({"Check": [key for key, value in resume_checks.items() if value is True],
                      "Status": ["PASS"] * sum(value is True for value in resume_checks.values())}))

## Overwrite protection and selection logic

A temporary directory tests incomplete-run protection, compatible completed-run skipping, rejection of incompatible completed runs, active locks, and archival overwrite behavior. Hand-verifiable labels check macro-F1, while synthetic scores check early-stopping ties and patience. None of these fixtures are saved as baseline performance.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
safety_checks = safety_logic_checks(info)
atomic_json(safety_checks, OUTPUT / "safety_logic_validation.json")
display(pd.DataFrame({"Check": [key for key, value in safety_checks.items() if value is True],
                      "Status": ["PASS"] * sum(value is True for value in safety_checks.values())}))

## Preparation audit and authorization boundary

Training functions require a matching successful preparation audit. Every one of the 25 intended identities shares the same scientific configuration fingerprint. Existing checkpoint/source hashes are checked again; the audit reads no main test prediction files. The automated runner also checks notebook language and leaves the full-training cell unexecuted.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
assert protected_hashes(ROOT) == protected_before, "Protected checkpoints or sources changed"
audit = {
    "configuration_fingerprint": info["configuration_fingerprint"],
    "trainer_sources": info["trainer_sources"], "runtime": info["runtime"],
    "all_adamw_preflights_passed": bool(memory.Status.eq("PASS").all()),
    "resume_validated": resume_checks["next_optimizer_step_bitwise_equal"],
    "overwrite_protection_validated": safety_checks["incomplete_run_protected"],
    "test_locked": resume_checks["test_loader_rejected_before_iteration"],
    "shared_fingerprint_for_25_runs": safety_checks["identical_configuration_fingerprint_for_25_runs"],
    "no_full_training_started": True,
    "protected_files_unchanged": True,
}
atomic_json(audit, OUTPUT / "preparation_audit.json")
atomic_json({"contract": info["contract"], "fingerprint": info["configuration_fingerprint"],
             "sources": info["trainer_sources"]}, OUTPUT / "training_configuration_fingerprint.json")
print("Preparation passed. Full training remains manually gated.")

## Manual training controls

The standalone cell below supports the user-authorized EfficientNet-B0 five-fold sequence. It always reloads the current implementation, verifies the preparation audit, and chooses each fold's action from saved files. Keep confirmation false for a read-only preview. Completed experiments are never overwritten.

### Standalone EfficientNet-B0 launch

The next manual cell is self-contained. It reloads the corrected source code even when this notebook kernel previously imported the older trainer. Set its confirmation flag to true only when ready to launch all five folds. Completed folds are preserved and skipped; checkpointed partial folds resume; metadata-only failed attempts are archived before a fresh start. No global overwrite flag is needed.

## MANUAL EXECUTION — FULL BASELINE TRAINING

This self-contained cell is the only full-training invocation. The preparation runner skips it. With confirmation true, folds 1–5 are handled sequentially under the unchanged protocol. A failed fold stops the sequence for review; no error is silently ignored.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
# MANUAL EXECUTION — FULL BASELINE TRAINING
from pathlib import Path
import importlib
import sys

ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
launcher = importlib.import_module("src.baselines.launch")
launcher = importlib.reload(launcher)

CONFIRM_FULL_TRAINING = False  # Change to True to start the five-fold sequence.
run_summary = launcher.train_model_folds(
    ROOT,
    model_name="efficientnet_b0",
    folds=(1, 2, 3, 4, 5),
    confirm_full_training=CONFIRM_FULL_TRAINING,
)
display(run_summary)

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
import sys
import importlib

ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

launcher = importlib.import_module("src.baselines.launch")
launcher = importlib.reload(launcher)

summary = launcher.train_model_folds(
    ROOT,
    model_name="efficientnet_b0",
    folds=(1, 2, 3, 4, 5),
    confirm_full_training=True,
)

display(summary)

## Completion summaries and saved training curves

After real runs complete, summary utilities save one row per model/fold with the best epoch, validation accuracy/F1, epoch count, early stopping, duration, and best checkpoint path. Model-wise validation means and sample SD (ddof=1) are written only after all five folds are complete. No test performance is included.

Each run saves a two-panel loss/F1 training curve to disk. The notebook does not display 25 large figures. Per-epoch times and cumulative compute time are reported separately from active run wall time; downtime between resumed sessions is excluded. The latter includes run I/O up to the last committed checkpoint and final completion, but may omit uncommitted work from an interrupted partial epoch.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
summary = completion_summary(ROOT, info)
if summary.empty:
    print("No baseline run has completed. No performance summary or training curves were fabricated.")
else:
    display(summary)

## Final preparation conclusions

All five architectures use the shared from-scratch protocol, one-channel MRI, three logits, and classification-only loss. Test data remain locked. Real AdamW memory safety, checkpoint restoration, next-update equivalence, overwrite protection, and configuration integrity have been checked with disposable objects.

**Preparation ends here. No full baseline training has started. The first real run requires manual execution of the labeled training cell with ResNet18 and Fold 1 selected.**

## Recovery note: AMP overflow (2026-10-06)

EfficientNet-B0 Fold 1 completed at epoch 55, with its best validation checkpoint at epoch 40. The saved traceback followed that completed run and originated in Fold 2 before its first completed epoch. The historical output above is retained for provenance.

The shared baseline optimizer routine now lets CUDA GradScaler skip a nonfinite scaled-gradient update and reduce its scale, instead of raising before the scaler can adapt. Nonfinite forward outputs, missing gradients, and nonfinite FP32 gradients still stop execution. Skipped updates are recorded per epoch. Hyperparameters, architecture, input data, and the initial scale remain unchanged.

Restart the kernel and run the imports cell to load the corrected module. The controls now select EfficientNet-B0 Fold 2 only, with confirmation disabled. Fold 2 has no checkpoint to resume; `OVERWRITE=True` archives its metadata-only failed attempt. Set confirmation to true yourself, then execute the manual training cell. Preserve the completed Fold 1. For any later interruption after a checkpoint exists, use `RESUME=True` and `OVERWRITE=False`.

The old trainer source hash is accepted only through an explicit compatibility rule; all other identity fields remain strict. Existing checkpoints and completed results are not rewritten. The repair report is saved under `results/baselines/training_setup/amp_overflow_repair/`.


**Notebook reload correction:** a subsequent traceback still referenced the old pre-fix function retained in kernel memory. The standalone launcher now explicitly reloads the trainer before use. Its five-fold plan supersedes the earlier Fold-2-only control instructions; there is no global overwrite flag. The saved error output is archived in `09_before_reload_fix.ipynb`.